In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
from loaders._load_vn30_multi_class_knn import preprocess, VN30

In [3]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import NeighborhoodComponentsAnalysis

In [4]:
class KNNWithMetricWindow:
    def __init__(self, n_neighbors=3, window_size=30, embedding_dim=None, random_state=42):
        """
        KNN với học hàm khoảng cách (NCA) và cửa sổ thời gian cố định.
        """
        self.n_neighbors = n_neighbors
        self.window_size = window_size
        self.embedding_dim = embedding_dim
        self.random_state = random_state
        
        self.nca = NeighborhoodComponentsAnalysis(
            n_components=self.embedding_dim,
            random_state=self.random_state
        )
        self.X_embedded = None
        self.y_hist = None

    def fit(self, X, y):
        # Học embedding từ toàn bộ train
        self.nca.fit(X, y)
        # Lưu embedding của X
        self.X_embedded = self.nca.transform(X)
        self.y_hist = np.array(y)

    def predict_with_window(self, X_future, y_future_hist):
        """
        X_future: feature cho đoạn thời gian sau train (ví dụ test)
        y_future_hist: nhãn thật cho đoạn sau train (dùng làm history khi predict)
        """
        # Transform toàn bộ X_future sang embedding space
        X_future_emb = self.nca.transform(X_future)

        preds = []
        # Lịch sử ban đầu = embedding của train tail
        X_all_emb = np.vstack([self.X_embedded[-self.window_size:], X_future_emb])
        y_all_hist = np.hstack([self.y_hist[-self.window_size:], y_future_hist])

        # Duyệt từng ngày dự đoán
        for t in range(self.window_size, len(X_all_emb)):
            X_window = X_all_emb[t-self.window_size:t]
            y_window = y_all_hist[t-self.window_size:t]
            x_query = X_all_emb[t].reshape(1, -1)

            knn_local = KNeighborsClassifier(n_neighbors=self.n_neighbors, metric="euclidean")
            knn_local.fit(X_window, y_window)
            preds.append(knn_local.predict(x_query)[0])

        return np.array(preds)

In [5]:
data = preprocess("ACB")
X_train, y_train = data["train"]
X_test, y_test = data["test"]

In [6]:
model = KNNWithMetricWindow(n_neighbors=3, window_size=30, embedding_dim=3)
model.fit(X_train, y_train)

In [8]:
y_pred = model.predict_with_window(X_test, y_test)

# Tính balanced accuracy
bal_acc = balanced_accuracy_score(y_test[model.window_size:], y_pred[model.window_size:])

print(bal_acc)

0.248474713692105


In [10]:
mean_accuracy = 0.0
for symbol in VN30:
    data = preprocess(symbol)
    X_train, y_train = data["train"]
    X_test, y_test = data["test"]

    model = KNNWithMetricWindow(window_size=30, embedding_dim=3)
    model.fit(X_train, y_train)

    y_pred = model.predict_with_window(X_test, y_test)

    bal_acc = balanced_accuracy_score(y_test[model.window_size:], y_pred[model.window_size:])
    print(f"Balanced accuracy for {symbol}: {bal_acc}")
    mean_accuracy += bal_acc

mean_accuracy /= len(VN30)
print(f"Mean balanced accuracy for VN30: {mean_accuracy}")

Balanced accuracy for ACB: 0.248474713692105
Balanced accuracy for BCM: 0.2812592592592592
Balanced accuracy for BID: 0.21479559585789976
Balanced accuracy for BVH: 0.21123233980494027
Balanced accuracy for CTG: 0.21867335590952797
Balanced accuracy for FPT: 0.22256775188282035
Balanced accuracy for GAS: 0.24408748114630469
Balanced accuracy for GVR: 0.22374286505940671
Balanced accuracy for HDB: 0.18987974987974987
Balanced accuracy for HPG: 0.19862990582630763
Balanced accuracy for LPB: 0.23205114055368264
Balanced accuracy for MBB: 0.1546378269617706
Balanced accuracy for MSN: 0.21077567047961782
Balanced accuracy for MWG: 0.22882543847749387
Balanced accuracy for PLX: 0.18584533193858108
Balanced accuracy for SAB: 0.20015015015015014
Balanced accuracy for SHB: 0.25913086913086913
Balanced accuracy for SSB: 0.20567044208987945
Balanced accuracy for SSI: 0.17910913647755752
Balanced accuracy for STB: 0.20495324283559574
Balanced accuracy for TCB: 0.20520810596189665
Balanced accuracy

In [11]:
from collections import Counter

class EnsembleMetricKNN:
    def __init__(self, configs, window_size=30, random_state=42):
        """
        configs: list các dict, mỗi dict chứa 'n_neighbors' và 'embedding_dim'
        """
        self.configs = configs
        self.window_size = window_size
        self.random_state = random_state
        self.models = []

    def fit(self, X, y):
        self.models = []
        for cfg in self.configs:
            model = KNNWithMetricWindow(
                n_neighbors=cfg["n_neighbors"],
                window_size=self.window_size,
                embedding_dim=cfg["embedding_dim"],
                random_state=self.random_state
            )
            model.fit(X, y)
            self.models.append(model)

    def predict_with_window(self, X_future, y_future_hist):
        all_preds = []
        for model in self.models:
            preds = model.predict_with_window(X_future, y_future_hist)
            all_preds.append(preds)
        
        # Chuyển sang np.array: shape (n_models, n_samples)
        all_preds = np.array(all_preds)
        # Lấy vote đa số cho từng sample
        final_preds = []
        for i in range(all_preds.shape[1]):
            votes = Counter(all_preds[:, i])
            final_preds.append(votes.most_common(1)[0][0])
        return np.array(final_preds)

In [ ]:
configs = [
    {"n_neighbors": 3, "embedding_dim": 3},
    {"n_neighbors": 5, "embedding_dim": 3},
    {"n_neighbors": 3, "embedding_dim": None},
    {"n_neighbors": 5, "embedding_dim": None}
]

In [14]:
mean_accuracy = 0.0
for symbol in VN30:
    data = preprocess(symbol)
    X_train, y_train = data["train"]
    X_test, y_test = data["test"]

    ensemble_model = EnsembleMetricKNN(configs=configs, window_size=30)
    ensemble_model.fit(X_train, y_train)
    y_pred = ensemble_model.predict_with_window(X_test, y_test)

    bal_acc = balanced_accuracy_score(y_test[ensemble_model.window_size:], y_pred[ensemble_model.window_size:])
    print(f"Balanced accuracy for {symbol}: {bal_acc}")
    mean_accuracy += bal_acc

mean_accuracy /= len(VN30)
print(f"Mean balanced accuracy for VN30: {mean_accuracy}")

Balanced accuracy for ACB: 0.23684199858112898
Balanced accuracy for BCM: 0.276973544973545
Balanced accuracy for BID: 0.21652866780612187
Balanced accuracy for BVH: 0.21266839850514993
Balanced accuracy for CTG: 0.2528591496300013
Balanced accuracy for FPT: 0.23891300480341573
Balanced accuracy for GAS: 0.237114127702363
Balanced accuracy for GVR: 0.205234969052639
Balanced accuracy for HDB: 0.16378066378066378
Balanced accuracy for HPG: 0.2103464501515476
Balanced accuracy for LPB: 0.24544170212434374
Balanced accuracy for MBB: 0.17094902749832325
Balanced accuracy for MSN: 0.2024349736520789
Balanced accuracy for MWG: 0.2164255999771719
Balanced accuracy for PLX: 0.1960768359049078
Balanced accuracy for SAB: 0.21111861861861864
Balanced accuracy for SHB: 0.2442688890057311
Balanced accuracy for SSB: 0.21642674461088784
Balanced accuracy for SSI: 0.17230120756436546
Balanced accuracy for STB: 0.19283257918552038
Balanced accuracy for TCB: 0.20242621782931308
Balanced accuracy for TPB